<img src="https://raw.githubusercontent.com/AnalyticJeremy/Olympics_Analysis/main/img/logos/olympics.svg" width="550" />

# Transform Data

Compile the raw data together into a dataset that is more conducive to analytics

In [0]:
library(SparkR)

In [0]:
countries <- read.df(path = "/olympics/raw/countries", source = "delta")
disciplines <- read.df(path = "/olympics/raw/disciplines", source = "delta")
edition_discipline_medal_summary <- read.df(path = "/olympics/raw/edition_discipline_medal_summary", source = "delta")
edition_disciplines <- read.df(path = "/olympics/raw/edition_disciplines", source = "delta")
edition_event_medals <- read.df(path = "/olympics/raw/edition_event_medals", source = "delta")
edition_events <- read.df(path = "/olympics/raw/edition_events", source = "delta")
edition_medal_summary <- read.df(path = "/olympics/raw/edition_medal_summary", source = "delta")
editions <- read.df(path = "/olympics/raw/editions", source = "delta")
display(editions)

#,Year,City,Country,Opened,Closed,Competition,Note,Url,Season,EditionID,ParticipantNote,MedalEventsNote,ParticipantCount,CountryCount,MedalEventCount,DisciplineCount
I,1896,Athina,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GRE.png,6 April,15 April,6 – 13 April,,/editions/1,Summer,1.0,176 from 13 countries,43 in 10 disciplines,176.0,13.0,43.0,10.0
II,1900,Paris,https://olympedia-flags.s3.eu-central-1.amazonaws.com/FRA.png,,,14 May – 28 October,,/editions/2,Summer,2.0,1239 from 27 countries,95 in 22 disciplines,1239.0,27.0,95.0,22.0
III,1904,St. Louis,https://olympedia-flags.s3.eu-central-1.amazonaws.com/USA.png,14 May,,1 July – 26 November,,/editions/3,Summer,3.0,650 from 10 countries,95 in 18 disciplines,650.0,10.0,95.0,18.0
IV,1908,London,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GBR.png,13 July,25 July,27 April – 31 October,,/editions/5,Summer,5.0,2025 from 23 countries,110 in 24 disciplines,2025.0,23.0,110.0,24.0
V,1912,Stockholm,https://olympedia-flags.s3.eu-central-1.amazonaws.com/SWE.png,6 July,15 July,5 May – 27 July,,/editions/6,Summer,6.0,2409 from 29 countries,107 in 19 disciplines,2409.0,29.0,107.0,19.0
VII,1920,Antwerpen,https://olympedia-flags.s3.eu-central-1.amazonaws.com/BEL.png,14 August,30 August,23 April – 12 September,,/editions/7,Summer,7.0,2680 from 29 countries,162 in 29 disciplines,2680.0,29.0,162.0,29.0
VIII,1924,Paris,https://olympedia-flags.s3.eu-central-1.amazonaws.com/FRA.png,5 July,27 July,4 May – 27 July,,/editions/8,Summer,8.0,3257 from 45 countries,131 in 23 disciplines,3257.0,45.0,131.0,23.0
IX,1928,Amsterdam,https://olympedia-flags.s3.eu-central-1.amazonaws.com/NED.png,28 July,12 August,17 May – 12 August,,/editions/9,Summer,9.0,3296 from 46 countries,125 in 20 disciplines,3296.0,46.0,125.0,20.0
X,1932,Los Angeles,https://olympedia-flags.s3.eu-central-1.amazonaws.com/USA.png,30 July,14 August,30 July – 14 August,,/editions/10,Summer,10.0,1924 from 47 countries,131 in 21 disciplines,1924.0,47.0,131.0,21.0
XI,1936,Berlin,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GER.png,1 August,16 August,1 – 16 August,,/editions/11,Summer,11.0,4483 from 49 countries,149 in 28 disciplines,4483.0,49.0,149.0,28.0


In [0]:
medals <- join(alias(edition_event_medals, "eem"), editions, edition_event_medals$EditionID == editions$EditionID, joinType="left_outer") |>
            filter(!isNull(edition_event_medals$Country)) |>
            select("Year", "Season", "DisciplineID", "Event", "Gender", "Medal", "eem.Country", "Participant")

medals <- join(alias(medals, "m"), disciplines, medals$DisciplineID == disciplines$DisciplineID, joinType="left_outer") |>
            select("Year", "m.Season", "m.DisciplineID", "Discipline", "Event", "Gender", "Medal", "Country", "Participant")

display(medals)

Year,Season,DisciplineID,Discipline,Event,Gender,Medal,Country,Participant
1896,Summer,GAR,Artistic Gymnastics,"Horse Vault, Men",Men,Gold,GER,Carl Schuhmann
1896,Summer,GAR,Artistic Gymnastics,"Horse Vault, Men",Men,Silver,SUI,Louis Zutter
1896,Summer,GAR,Artistic Gymnastics,"Horse Vault, Men",Men,Bronze,GER,Hermann Weingärtner
1896,Summer,GAR,Artistic Gymnastics,"Parallel Bars, Men",Men,Gold,GER,Alfred Flatow
1896,Summer,GAR,Artistic Gymnastics,"Parallel Bars, Men",Men,Silver,SUI,Louis Zutter
1896,Summer,GAR,Artistic Gymnastics,"Parallel Bars, Teams, Men",Men,Gold,GER,Germany
1896,Summer,GAR,Artistic Gymnastics,"Parallel Bars, Teams, Men",Men,Silver,GRE,Panellinios Gymnastikos Syllogos
1896,Summer,GAR,Artistic Gymnastics,"Parallel Bars, Teams, Men",Men,Bronze,GRE,Ethnikos Gymnastikos Syllogos
1896,Summer,GAR,Artistic Gymnastics,"Horizontal Bar, Men",Men,Gold,GER,Hermann Weingärtner
1896,Summer,GAR,Artistic Gymnastics,"Horizontal Bar, Men",Men,Silver,GER,Alfred Flatow


In [0]:
dbutils.fs.rm("/olympics/medals", recurse = TRUE);
write.df(medals, path=paste0("/olympics/medals"), source="delta", mode="overwrite")

In [0]:
medal_table <- medals |>
                  group_by("Year", "Season", "DisciplineID", "Gender", "Country", "Medal") |>
                  count() |>
                  group_by("Year", "Season", "DisciplineID", "Gender", "Country") |>
                  pivot("Medal", c("Gold", "Silver", "Bronze")) |>
                  sum("count") |>
                  fillna(0)

medal_table <- join(alias(medal_table, "mt"), disciplines, medal_table$DisciplineID == disciplines$DisciplineID, joinType="left_outer") |>
                  select("Year", "mt.Season", "mt.DisciplineID", "Discipline", "Gender", "Country", "Gold", "Silver", "Bronze")

medal_table <- medal_table |> withColumn("Total", medal_table$Gold + medal_table$Silver + medal_table$Bronze)

# Weighted Score - This is an attempt to provide a score for medal performance by assigning weights to the medals... gold is more valuable than silver
medal_table <- medal_table |> withColumn("Weighted", (medal_table$Gold * 7) + (medal_table$Silver * 4) + (medal_table$Bronze * 2))

medal_table <- medal_table |> arrange(medal_table$Year, medal_table$Season, medal_table$DisciplineID, medal_table$Gender, desc(medal_table$Total), desc(medal_table$Weighted), medal_table$Country)

display(medal_table)

Year,Season,DisciplineID,Discipline,Gender,Country,Gold,Silver,Bronze,Total,Weighted
1896,Summer,ATH,Athletics,Men,USA,9,6,2,17,91.0
1896,Summer,ATH,Athletics,Men,GRE,1,3,6,10,31.0
1896,Summer,ATH,Athletics,Men,HUN,0,1,2,3,8.0
1896,Summer,ATH,Athletics,Men,AUS,2,0,0,2,14.0
1896,Summer,ATH,Athletics,Men,FRA,0,1,1,2,6.0
1896,Summer,ATH,Athletics,Men,GBR,0,1,1,2,6.0
1896,Summer,ATH,Athletics,Men,GER,0,1,0,1,4.0
1896,Summer,CRD,Cycling Road,Men,GRE,1,0,0,1,7.0
1896,Summer,CRD,Cycling Road,Men,GER,0,1,0,1,4.0
1896,Summer,CRD,Cycling Road,Men,GBR,0,0,1,1,2.0


In [0]:
dbutils.fs.rm("/olympics/medal_table", recurse = TRUE);
write.df(medal_table, path=paste0("/olympics/medal_table"), source="delta", mode="overwrite")